# Offset vs inherent disagreement between BMRB depositions, part 2

Offset and inherent RMSE of the filtered pairs. Run `offset_vs_inherent_1_decomposition.ipynb` and `binding_vs_corr_rmse.ipynb` first.

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def set_style(base=14, family="Arial"):
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base + 2, "axes.labelsize": base,
        "xtick.labelsize": base - 2, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",   # keep text as text in SVG
    })

colors = {"LACS": "#E06C75", "PANAV": "#61AFEF"}
FIGS = "Figs"
os.makedirs(FIGS, exist_ok=True)

DECOMP = "../../data/analysis/Fig3/inherent_test/decomposition_no_othernoise.json"          # pairs with state differences removed
D = pd.DataFrame(json.load(open(DECOMP)))
print(f"{len(D)} records | " + " ".join(f"{m}={n}" for m, n in D.method.value_counts().items()))

# ===== inherent RMSE (= sqrt(inherent_mse)) distribution, LACS and PANAV overlaid =====
set_style(16)
SUBSET = "consistent"          # consistent subset; set to "all" to use every pair

fig, ax = plt.subplots(figsize=(6.5, 5.2))

# Both methods share the same bin edges; otherwise hist(bins=30) bins each by its own min/max
# and the overlaid bars would not line up
subs = {}
for method in ["PANAV", "LACS"]:
    if SUBSET == "consistent":
        sub = D[(D.method == method) & D.consistent]
    else:
        sub = D[D.method == method]
    subs[method] = np.sqrt(sub["inherent_mse"].values)     # == sub["corr_rmse"]

all_vals = np.concatenate(list(subs.values()))
bins = np.linspace(all_vals.min(), all_vals.max(), 31)

for method in ["PANAV", "LACS"]:
    vals = subs[method]
    ax.hist(vals, bins=bins, alpha=0.5, color=colors[method],
            edgecolor="white",
            label=f"{method}")
    print(f"n={len(vals)}, median={np.median(vals):.2f}")

ax.set_xlabel("Protein level inherent RMSE (ppm)")
ax.set_ylabel("Count")
from matplotlib.ticker import MultipleLocator
ax.set_ylim(0, 30)
ax.yaxis.set_major_locator(MultipleLocator(4))

ax.legend(frameon=False, loc="upper right")
# ax.grid(axis="y", alpha=0.3, ls="--")
plt.tight_layout()
plt.savefig(f"{FIGS}/inherent_rmse_distribution_no_othernoise.svg", bbox_inches="tight")
plt.savefig(f"{FIGS}/inherent_rmse_distribution_no_othernoise.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
set_style(16)
groups = ["LACS", "PANAV"]
x = np.arange(len(groups))

off_rmse, inh_rmse, ns = [], [], []
for m in groups:
    sub = D[(D.method == m) & D.consistent]
    off_rmse.append(np.sqrt(np.mean(sub["offset"].values ** 2)))       # RMS of the per-pair offset
    inh_rmse.append(np.sqrt(np.mean(sub["corr_rmse"].values ** 2)))    # RMS of the per-pair inherent RMSE
    ns.append(len(sub))
off_rmse = np.array(off_rmse)
inh_rmse = np.array(inh_rmse)

fig, ax = plt.subplots(figsize=(6.5, 5.2))
w = 0.38
ax.bar(x - w/2, off_rmse, w, color="#888888", edgecolor="white", label="offset")
ax.bar(x + w/2, inh_rmse, w, color="#C678DD", edgecolor="white", label="intrinsic RMSE")
for xi, v in zip(x - w/2, off_rmse):
    ax.text(xi, v, f"{v:.2f}", ha="center", va="bottom")
    print(f"{xi:.2f} {v:.2f}")
for xi, v in zip(x + w/2, inh_rmse):
    ax.text(xi, v, f"{v:.2f}", ha="center", va="bottom")
    print(f"{xi:.2f} {v:.2f}")
ax.set_xticks(x)
ax.set_xticklabels([f"{l}\n(n={n})" for l, n in zip(groups, ns)])
ax.set_ylabel("RMSE (ppm)")
ax.set_ylim(0, max(off_rmse.max(), inh_rmse.max()) * 1.4)
ax.legend(frameon=False, loc="upper left")
# ax.grid(axis="y", alpha=0.3, ls="--")
plt.tight_layout()
# plt.savefig(f"{FIGS}/rmse_absolute_consistent.svg", bbox_inches="tight")
# plt.savefig(f"{FIGS}/rmse_absolute_consistent.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
set_style(16)
m = "PANAV"
sub = D[(D.method == m) & D.consistent]
off = np.sqrt(np.mean(sub["offset"].values ** 2))       # RMS of the per-pair offset
inh = np.sqrt(np.mean(sub["corr_rmse"].values ** 2))    # RMS of the per-pair inherent RMSE
n = len(sub)

labels = ["Offset", "Intrinsic\nRMSE"]
vals = [off, inh]
x = np.arange(len(vals))

fig, ax = plt.subplots(figsize=(3.8, 5.2))
ax.bar(x, vals, 0.8, color=["#888888", "#C678DD"], edgecolor="white")
for xi, v in zip(x, vals):
    # ax.text(xi, v, f"{v:.2f}", ha="center", va="bottom")
    print(f"{labels[xi]!r}: {v:.3f}")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlim(-0.6, len(vals) - 0.4)                      # keep bars from stretching across the canvas
ax.set_ylabel("RMSE (ppm)")
ax.set_ylim(0, max(vals) * 1.25)
plt.tight_layout()
plt.savefig(f"{FIGS}/rmse_absolute_consistent_panav.svg", bbox_inches="tight")
plt.savefig(f"{FIGS}/rmse_absolute_consistent_panav.png", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
set_style(16)
m = "LACS"
sub = D[(D.method == m) & D.consistent]
off = np.sqrt(np.mean(sub["offset"].values ** 2))       # RMS of the per-pair offset
inh = np.sqrt(np.mean(sub["corr_rmse"].values ** 2))    # RMS of the per-pair inherent RMSE
n = len(sub)

labels = ["Offset", "Intrinsic\nRMSE"]
vals = [off, inh]
x = np.arange(len(vals))

fig, ax = plt.subplots(figsize=(3.8, 5.2))
ax.bar(x, vals, 0.8, color=["#888888", "#C678DD"], edgecolor="white")
for xi, v in zip(x, vals):
    # ax.text(xi, v, f"{v:.2f}", ha="center", va="bottom")
    print(f"{labels[xi]!r}: {v:.3f}")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlim(-0.6, len(vals) - 0.4)                      # keep bars from stretching across the canvas
ax.set_ylabel("RMSE (ppm)")
ax.set_ylim(0, max(vals) * 1.25)
plt.tight_layout()
plt.savefig(f"{FIGS}/rmse_absolute_consistent_lacs.svg", bbox_inches="tight")
plt.savefig(f"{FIGS}/rmse_absolute_consistent_lacs.png", bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.ticker import MultipleLocator
from scipy.stats import gaussian_kde

def set_style(base=14, family="Arial"):
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base + 2, "axes.labelsize": base,
        "xtick.labelsize": base - 2, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",   # keep text as text in SVG
    })

LIGHT_BLUE, DARK_BLUE = "#A9CCE3", "#3F729B"
FIGS = "Figs"
os.makedirs(FIGS, exist_ok=True)
METHOD = "PANAV"
KDE_BW = 0.2        # gaussian_kde bandwidth factor; larger = smoother curve

# two stages of the pair filter: automatic only (near-duplicate + classifier) vs automatic + manual review
SOURCES = {
    "- manual review": ("../../data/analysis/Fig3/inherent_test/decomposition_auto_only.json",     LIGHT_BLUE),
    "+ manual review":  ("../../data/analysis/Fig3/inherent_test/decomposition_no_othernoise.json", DARK_BLUE),
}
D_auto = pd.DataFrame(json.load(open(SOURCES["- manual review"][0])))
D      = pd.DataFrame(json.load(open(SOURCES["+ manual review"][0])))

subs = {lab: np.sqrt(df[(df.method == METHOD) & df.consistent]["inherent_mse"].values)   # == corr_rmse
        for lab, df in (("- manual review", D_auto), ("+ manual review", D))}

# the manual-review set must be a subset of the automatic set (true while MANUAL_KEEP is empty)
key = lambda df: set(zip(df[df.method == METHOD].eid_a, df[df.method == METHOD].eid_b))
assert key(D) <= key(D_auto), "manual-review records missing from the automatic set"

# ===== inherent RMSE distribution, PANAV only, two filter stages overlaid =====
set_style(16)
fig, ax = plt.subplots(figsize=(7, 5.2))

all_vals = np.concatenate(list(subs.values()))
bins = np.linspace(all_vals.min(), all_vals.max(), 31)   # shared bin edges for both stages
bin_w = bins[1] - bins[0]

# draw the larger (automatic) set first; the subset sits on top, so the visible light-blue part
# of each bin is what the manual review removed
for lab, vals in subs.items():
    ax.hist(vals, bins=bins, color=SOURCES[lab][1], edgecolor="white",
            label=f"{lab} (n={len(vals)})")
    print(f"{lab}: n={len(vals)}, median={np.median(vals):.2f}")

# KDE curves scaled to counts (density * n * bin width). RMSE is bounded at 0, so the KDE is
# reflected at x = 0 (k(x) + k(-x)) to keep the mass from leaking into negative values.
xg = np.linspace(0, bins[-1], 400)
for lab, vals in subs.items():
    kde = gaussian_kde(vals, bw_method=KDE_BW)
    ax.plot(xg, (kde(xg) + kde(-xg)) * len(vals) * bin_w, color=SOURCES[lab][1], lw=4, zorder=3,
            path_effects=[pe.Stroke(linewidth=4.6, foreground="white"), pe.Normal()])   # white halo over bars

ax.set_xlabel("Protein level intrinsic RMSE (ppm)")
ax.set_ylabel("Count")
ax.set_ylim(0, 48)
ax.yaxis.set_major_locator(MultipleLocator(8))
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.savefig(f"{FIGS}/inherent_rmse_distribution_panav_auto_vs_manual.svg", bbox_inches="tight")
plt.savefig(f"{FIGS}/inherent_rmse_distribution_panav_auto_vs_manual.png", bbox_inches="tight", dpi=300)
plt.show()
